# Nonlinear Spatial Panel Models

This guide demonstrates `SARPanelTobit` and `SEMPanelTobit` on synthetic panel data.

Both models handle **censored outcomes** observed for $N$ spatial units across $T$ time periods.

- **SARPanelTobit** — spatial lag in the latent outcome: $y^*_t = \rho W y^*_t + X_t \beta + \varepsilon_t$
- **SEMPanelTobit** — spatial lag in the disturbance: $y^*_t = X_t \beta + u_t$, $u_t = \lambda W u_t + \varepsilon_t$

Observed outcomes are censored at zero: $y_t = \max(0, y^*_t)$.

In [ ]:
import numpy as np
from libpysal.graph import Graph
from bayespecon import SARPanelTobit, SEMPanelTobit

rng = np.random.default_rng(42)

# Panel dimensions: 3x3 rook grid, 10 time periods
N = 9
T = 10
side = 3

# Build row-standardised rook-contiguity weights
W_dense = np.zeros((N, N))
for r in range(side):
    for c in range(side):
        i = r * side + c
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < side and 0 <= nc < side:
                W_dense[i, nr * side + nc] = 1.0
row_sums = W_dense.sum(axis=1, keepdims=True)
W_dense /= np.where(row_sums == 0, 1, row_sums)

# Convert to libpysal Graph
i_idx, j_idx = np.where(W_dense != 0)
W_graph = Graph.from_arrays(i_idx, j_idx, W_dense[i_idx, j_idx].astype(float))

## SAR Panel Tobit

Generate censored panel data from a spatial-lag process and fit `SARPanelTobit`.

In [ ]:
rho_true = 0.35
beta_true = np.array([1.0, 1.4])
sigma_true = 0.8

A_inv = np.linalg.inv(np.eye(N) - rho_true * W_dense)

y_list, X_list = [], []
for _ in range(T):
    Xt = np.column_stack([np.ones(N), rng.standard_normal(N)])
    eps = sigma_true * rng.standard_normal(N)
    y_lat = A_inv @ (Xt @ beta_true + eps)
    y_list.append(np.maximum(0.0, y_lat))
    X_list.append(Xt)

y_sar = np.concatenate(y_list)
X_sar = np.vstack(X_list)

print(f"Censored fraction: {(y_sar == 0).mean():.2f}")

sar_model = SARPanelTobit(y=y_sar, X=X_sar, W=W_graph, N=N, T=T, censoring=0.0)
sar_idata = sar_model.fit(tune=500, draws=500, chains=2, random_seed=42, progressbar=False)

rho_hat = float(sar_idata.posterior["rho"].mean())
beta_hat = sar_idata.posterior["beta"].mean(("chain", "draw")).values
print(f"rho:   true={rho_true:.2f}  estimated={rho_hat:.3f}")
for k, (bt, bh) in enumerate(zip(beta_true, beta_hat)):
    print(f"beta[{k}]: true={bt:.2f}  estimated={bh:.3f}")

## SEM Panel Tobit

Generate censored panel data from a spatial-error process and fit `SEMPanelTobit`.

In [ ]:
lam_true = 0.35

B_inv = np.linalg.inv(np.eye(N) - lam_true * W_dense)

y_list, X_list = [], []
for _ in range(T):
    Xt = np.column_stack([np.ones(N), rng.standard_normal(N)])
    u_t = B_inv @ (sigma_true * rng.standard_normal(N))
    y_lat = Xt @ beta_true + u_t
    y_list.append(np.maximum(0.0, y_lat))
    X_list.append(Xt)

y_sem = np.concatenate(y_list)
X_sem = np.vstack(X_list)

print(f"Censored fraction: {(y_sem == 0).mean():.2f}")

sem_model = SEMPanelTobit(y=y_sem, X=X_sem, W=W_graph, N=N, T=T, censoring=0.0)
sem_idata = sem_model.fit(tune=500, draws=500, chains=2, random_seed=42, progressbar=False)

lam_hat = float(sem_idata.posterior["lam"].mean())
beta_hat = sem_idata.posterior["beta"].mean(("chain", "draw")).values
print(f"lam:   true={lam_true:.2f}  estimated={lam_hat:.3f}")
for k, (bt, bh) in enumerate(zip(beta_true, beta_hat)):
    print(f"beta[{k}]: true={bt:.2f}  estimated={bh:.3f}")